# Final Forecast Walkthrough

This notebook is the presentation layer for the final tuned Barents Watch lice forecasting system.

What this notebook covers:

- data scope and leakage control,
- the final model policy by horizon,
- the feature design,
- candidate model selection,
- high-risk sites and error hotspots,
- and what to say about deep learning, extra features, and external datasets.

Important: no training example is allowed to use a target window that extends into 2026.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

ROOT

In [ ]:
from lice.config import PROCESSED_DIR, RESULTS_DIR, TRAINING_MAX_YEAR

artifact_paths = {
    'audit': RESULTS_DIR / 'data_audit.json',
    'metrics': RESULTS_DIR / 'model_metrics.json',
    'comparison': RESULTS_DIR / 'model_comparison.csv',
    'errors': RESULTS_DIR / 'error_summary_by_area.csv',
    'predictions': RESULTS_DIR / 'holdout_predictions.csv',
    'features': RESULTS_DIR / 'feature_columns.csv',
    'documentation': RESULTS_DIR / 'final_model_documentation.md',
    'master': PROCESSED_DIR / 'master_table.parquet',
}

with open(artifact_paths['audit'], 'r', encoding='utf-8') as handle:
    audit_summary = json.load(handle)

with open(artifact_paths['metrics'], 'r', encoding='utf-8') as handle:
    metrics = json.load(handle)

with open(artifact_paths['documentation'], 'r', encoding='utf-8') as handle:
    final_model_documentation = handle.read()

comparison = pd.read_csv(artifact_paths['comparison'])
errors = pd.read_csv(artifact_paths['errors'])
predictions = pd.read_csv(artifact_paths['predictions'], parse_dates=['week_start_date'])
feature_columns = pd.read_csv(artifact_paths['features'])['feature'].tolist()
latest_validated_week = predictions['week_start_date'].max()

master_preview = pd.read_parquet(
    artifact_paths['master'],
    columns=[
        'sitenumber',
        'sitename',
        'week_start_date',
        'productionarea',
        'femaleadult',
        'mobilelice',
        'seatemperature',
        'treatment_count',
    ],
)
master_preview = (
    master_preview[master_preview['week_start_date'] <= latest_validated_week]
    .dropna(subset=['femaleadult'])
    .sort_values('week_start_date', ascending=False)
    .head(12)
)

{
    'artifacts_loaded': len(artifact_paths),
    'feature_count': len(feature_columns),
    'latest_prediction_week': latest_validated_week.date().isoformat(),
    'training_max_year': TRAINING_MAX_YEAR,
}

## 1. Executive Summary

This notebook is designed for presentation, not for model development. It mostly focuses on the classifier.

In [ ]:
from IPython.display import HTML

horizon_order = {"1w": 1, "2w": 2, "12w": 12}

metrics_rows = []
for horizon, horizon_metrics in metrics.items():
    for model_type, values in horizon_metrics.items():
        row = {"horizon": horizon, "model_type": model_type}
        row.update(values)
        metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
metrics_df["horizon_order"] = metrics_df["horizon"].map(horizon_order)

classifier_summary = (
    metrics_df[metrics_df["model_type"] == "classifier_any"][
        [
            "horizon",
            "selected_model",
            "decision_threshold",
            "holdout_positive_rate",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "horizon_order",
        ]
    ]
    .sort_values("horizon_order")
    .drop(columns="horizon_order")
    .reset_index(drop=True)
)


classifier_selection_notes = pd.DataFrame([
    {
        'horizon': '1w',
        'selected_classifier': 'hist_gb_default',
        'why_selected': 'Best short-horizon holdout F1 and PR-AUC. The simpler boosted-tree model beat the XGBoost variants here.',
    },
    {
        'horizon': '2w',
        'selected_classifier': 'hist_gb_balanced',
        'why_selected': 'A direct holdout audit showed it beat both XGBoost classifier variants for 2-week F1.',
    },
    {
        'horizon': '12w',
        'selected_classifier': 'xgb_gpu_balanced_baseline',
        'why_selected': 'The baseline XGBoost classifier recovered the best long-horizon recall and holdout F1 once near ties favored recall.',
    },
])

display(HTML(classifier_selection_notes.to_html(index=False)))
classifier_summary.round(4)

## 2. Data Scope And Leakage Guard

The most important modeling rule is that training must not use any example whose future target window reaches into 2026.

That means:

- the raw files may contain rows from 2026,
- but the supervised rows used for model fitting are only eligible when the full prediction window ends in 2025 or earlier,
- and all time-dependent features are lagged or built from lagged series.

In [ ]:
audit_table = pd.DataFrame([
    {'metric': 'Raw lice rows', 'value': audit_summary['vlice_rows']},
    {'metric': 'Raw treatment rows', 'value': audit_summary['vtreatment_rows']},
    {'metric': 'Unique lice sites', 'value': audit_summary['vlice_unique_sites']},
    {'metric': 'Unique treatment sites', 'value': audit_summary['vtreatment_unique_sites']},
    {'metric': 'Production areas', 'value': audit_summary['vlice_unique_production_areas']},
    {'metric': 'Counted lice weeks', 'value': audit_summary['vlice_counted_weeks']},
    {'metric': 'Likely no fish weeks', 'value': audit_summary['vlice_likely_no_fish_weeks']},
    {'metric': 'Raw lice year range', 'value': f"{audit_summary['vlice_min_year']} to {audit_summary['vlice_max_year']}"},
    {'metric': 'Raw treatment year range', 'value': f"{audit_summary['vtreatment_min_year']} to {audit_summary['vtreatment_max_year']}"},
])

leakage_guard_df = pd.DataFrame([
    {
        'horizon': horizon,
        'train_end': horizon_metrics['classifier_any']['train_end'],
        'calibration_end': horizon_metrics['classifier_any']['calibration_end'],
        'holdout_end': horizon_metrics['classifier_any']['holdout_end'],
        'eligible_window_rule': 'Target window must end in 2025 or earlier',
    }
    for horizon, horizon_metrics in metrics.items()
])
leakage_guard_df['horizon_order'] = leakage_guard_df['horizon'].map({'1w': 1, '2w': 2, '12w': 12})
leakage_guard_df = leakage_guard_df.sort_values('horizon_order').drop(columns='horizon_order').reset_index(drop=True)

display(audit_table)
display(leakage_guard_df)
master_preview

## 3. Feature Design And Model Selection

The final model uses 116 numeric features. The design is dominated by five ideas:

- recent site lice pressure,
- recent site treatment activity,
- site-level temporal history,
- production-area context,
- and neighbor pressure within 50 km.

Every spatial and temporal feature is built so it only uses information available at prediction time.

In [ ]:
from IPython.display import HTML

def classify_feature_group(feature: str) -> str:
    current_site_state = {
        'femaleadult',
        'mobilelice',
        'persistentlice',
        'seatemperature',
        'licelimitweek',
        'femaleadult_to_limit_ratio',
        'mobile_to_female_ratio',
        'persistent_to_female_ratio',
    }
    static_metadata = {'latitude', 'longitude', 'productionareaid', 'countynumber', 'year_index'}
    seasonal_and_availability = {
        'week_sin',
        'week_cos',
        'month',
        'quarter',
        'likelynofish_flag',
        'havecountedlice_flag',
        'seatemperature_missing',
    }
    site_history = {
        'site_breach_rate_expanding',
        'site_limit_ratio_expanding',
        'weeks_since_last_counted',
        'weeks_since_last_breach',
        'weeks_since_any_treatment',
        'weeks_since_medicinal_treatment',
        'weeks_since_mechanical_treatment',
    }
    current_treatment = {
        'treatment_count',
        'any_treatment',
        'mechanical_treatment_count',
        'medicinal_treatment_count',
        'bath_treatment_count',
        'feed_treatment_count',
        'full_site_treatment_count',
        'partial_site_treatment_count',
        'cleanerfish_treatment_count',
        'active_ingredient_count',
    }

    if feature in current_site_state:
        return 'Current site state'
    if feature in static_metadata:
        return 'Static metadata'
    if feature in seasonal_and_availability:
        return 'Seasonality and availability'
    if feature in site_history:
        return 'Site history'
    if feature in current_treatment:
        return 'Current treatment'
    if feature.startswith('pa_'):
        return 'Production-area context'
    if feature.startswith('neighbor_'):
        return 'Neighbor pressure'
    if any(feature.startswith(prefix) for prefix in [
        'femaleadult_lag_',
        'mobilelice_lag_',
        'persistentlice_lag_',
        'seatemperature_lag_',
        'breach_this_week_lag_',
        'femaleadult_to_limit_ratio_lag_',
    ]):
        return 'Site lags'
    if any(feature.startswith(prefix) for prefix in [
        'treatment_count_roll',
        'mechanical_treatment_count_roll',
        'medicinal_treatment_count_roll',
        'any_treatment_roll',
    ]):
        return 'Treatment rolling'
    if any(feature.endswith(suffix) for suffix in ['_roll4_mean', '_roll12_max', '_roll12_std', '_roll4_sum', '_roll12_sum', '_roll26_sum']):
        return 'Site rolling'
    return 'Other'

feature_df = pd.DataFrame({'feature': feature_columns})
feature_df['feature_group'] = feature_df['feature'].map(classify_feature_group)

feature_group_summary = (
    feature_df.groupby('feature_group', as_index=False)
    .agg(feature_count=('feature', 'size'))
    .sort_values('feature_count', ascending=False)
    .reset_index(drop=True)
)

feature_examples = (
    feature_df.groupby('feature_group')['feature']
    .apply(lambda series: ', '.join(series.head(6)))
    .reset_index(name='example_features')
)

display(HTML(feature_group_summary.to_html(index=False)))
display(HTML(feature_examples.to_html(index=False)))

In [ ]:
comparison['rank_within_task'] = comparison.groupby(['task_type', 'horizon'])['selection_score'].rank(ascending=False, method='first')

classifier_comparison = comparison[comparison['task_type'] == 'classifier_any'].copy()
selected_candidates = classifier_comparison[classifier_comparison['selected']].sort_values(['horizon']).reset_index(drop=True)
leaderboard = classifier_comparison[classifier_comparison['rank_within_task'] <= 3].sort_values(['horizon', 'rank_within_task']).reset_index(drop=True)

display(
    selected_candidates[
        [
            'horizon',
            'candidate_model',
            'selection_score',
            'calibration_threshold',
            'calibration_recall',
            'calibration_f1',
            'calibration_pr_auc',
        ]
    ].round(4)
)

leaderboard[
    [
        'horizon',
        'candidate_model',
        'selected',
        'rank_within_task',
        'selection_score',
        'calibration_threshold',
        'calibration_recall',
        'calibration_f1',
        'calibration_pr_auc',
    ]
] .round(4)

## 3B. What Drives The Final Classifiers

These charts focus on the selected binary breach classifier for each horizon because that is the clearest operational model.

How to read them:

- importance is measured on the valid holdout slice,
- a larger bar means model ranking quality drops more when that feature is shuffled,
- the score is average precision, so the plots are aligned with rare-event ranking quality,
- and the charts are explanatory, not causal.

In [ ]:
import pickle

import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

from lice.model import build_feature_matrix, split_train_calibration_holdout

importance_columns = sorted(
    set(
        feature_columns
        + [
            'week_start_date',
            'havecountedlice',
            'target_1w_any',
            'target_2w_any',
            'target_12w_any',
            'target_1w_pre_2026_only',
            'target_2w_pre_2026_only',
            'target_12w_pre_2026_only',
        ]
    )
)
importance_frame = pd.read_parquet(artifact_paths['master'], columns=importance_columns)

selected_classifier_specs = [
    {
        'horizon': 1,
        'label_column': 'target_1w_any',
        'model_path': RESULTS_DIR / 'models' / 'classifier_1w.pkl',
        'title': '1w selected classifier',
        'color': '#1f77b4',
    },
    {
        'horizon': 2,
        'label_column': 'target_2w_any',
        'model_path': RESULTS_DIR / 'models' / 'classifier_2w.pkl',
        'title': '2w selected classifier',
        'color': '#ff7f0e',
    },
    {
        'horizon': 12,
        'label_column': 'target_12w_any',
        'model_path': RESULTS_DIR / 'models' / 'classifier_12w.pkl',
        'title': '12w selected classifier',
        'color': '#2ca02c',
    },
]

top_n = 10
sample_size = 1200
importance_tables = []

fig, axes = plt.subplots(1, len(selected_classifier_specs), figsize=(18, 8))

for ax, spec in zip(axes, selected_classifier_specs):
    with open(spec['model_path'], 'rb') as handle:
        model = pickle.load(handle)

    holdout = split_train_calibration_holdout(
        importance_frame, spec['label_column'], spec['horizon']
    )['holdout']
    if len(holdout) > sample_size:
        holdout = holdout.sample(n=sample_size, random_state=42)

    X_holdout = build_feature_matrix(holdout, feature_columns)
    y_holdout = holdout[spec['label_column']].astype(int)

    result = permutation_importance(
        model,
        X_holdout,
        y_holdout,
        scoring='average_precision',
        n_repeats=3,
        random_state=42,
        n_jobs=1,
    )

    importance = pd.DataFrame(
        {
            'feature': feature_columns,
            'importance_mean': result.importances_mean,
            'importance_std': result.importances_std,
            'horizon': f"{spec['horizon']}w",
        }
    ).sort_values('importance_mean', ascending=False)

    top_features = importance.head(top_n).sort_values('importance_mean')
    importance_tables.append(top_features.sort_values('importance_mean', ascending=False))

    ax.barh(
        top_features['feature'],
        top_features['importance_mean'],
        xerr=top_features['importance_std'],
        color=spec['color'],
        alpha=0.85,
    )
    ax.set_title(spec['title'])
    ax.set_xlabel('Drop in average precision when shuffled')
    ax.grid(axis='x', alpha=0.2)

fig.suptitle('Top feature drivers of the final breach classifiers', fontsize=15, y=1.02)
plt.tight_layout()

classifier_feature_importance = pd.concat(importance_tables, ignore_index=True)
classifier_feature_importance[['horizon', 'feature', 'importance_mean', 'importance_std']].round(4)

In [ ]:
risk_12w = predictions[(predictions['model_type'] == 'classifier_any') & (predictions['horizon'] == 12)].copy()
risk_12w['week_label'] = risk_12w['week_start_date'].dt.strftime('%G-W%V')

area_risk_12w = (
    risk_12w.groupby('productionarea', as_index=False)
    .agg(
        mean_score=('score', 'mean'),
        actual_breach_rate=('actual', 'mean'),
        predicted_positive_rate=('prediction', 'mean'),
        site_weeks=('score', 'size'),
    )
    .sort_values('mean_score', ascending=False)
    .reset_index(drop=True)
)

top_risk_sites_12w = (
    risk_12w.sort_values('score', ascending=False)[
        ['week_label', 'productionarea', 'sitename', 'actual', 'prediction', 'score', 'candidate_model']
    ]
    .head(20)
    .reset_index(drop=True)
)

display(area_risk_12w.round(4))
top_risk_sites_12w.round(4)

In [ ]:
error_hotspots = (
    errors.groupby(['horizon', 'error_type', 'productionarea'], as_index=False)
    .agg(total_errors=('count', 'sum'), mean_score=('average_score', 'mean'))
    .sort_values(['horizon', 'error_type', 'total_errors'], ascending=[True, True, False])
    .reset_index(drop=True)
)

top_error_hotspots = error_hotspots.groupby(['horizon', 'error_type']).head(8).reset_index(drop=True)

top_12w_false_negatives = (
    risk_12w[(risk_12w['actual'] == 1) & (risk_12w['prediction'] == 0)]
    .sort_values('score', ascending=False)[['week_label', 'productionarea', 'sitename', 'score']]
    .head(15)
    .reset_index(drop=True)
)

display(top_error_hotspots.round(4))
top_12w_false_negatives.round(4)